In [0]:
%pip install aiohttp

In [0]:
import asyncio
import aiohttp
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData
import json

In [0]:
SECRET_SCOPE = "default2"
personal_conn_string = dbutils.secrets.get(scope=SECRET_SCOPE, key="blech-conn-string-evh")
evh_name = "blech_evh"

In [0]:
async def run(session, url):
    # Creating a producer client to send messages to the event hub.
    # Specifying a connection string to your event hubs namespace and
    # the event hub name.
    producer = EventHubProducerClient.from_connection_string(
        conn_str=personal_conn_string,
        eventhub_name=evh_name,
    )

    async with producer:
        # Creating a batch.
        batch = await producer.create_batch()
        last_sent_time = asyncio.get_event_loop().time()          

        async with session.get(url) as response:        
            async for line in response.content:  
                decoded_line = line.decode('utf-8').strip()   # line decoded to string
                if decoded_line.startswith("data:"):
                    cut_line = decoded_line[6:]               # I need to transform string to get data in better format       

                    try:
                        batch.add(EventData(cut_line))
                    except ValueError:                             # raises ValueError if max batch size has been exceeded
                        print(f"Number of elements in one batch: {len(batch)}")
                        await producer.send_batch(batch)
                        batch = await producer.create_batch()
                        batch.add(EventData(cut_line))
                        last_sent_time = asyncio.get_event_loop().time() 
                    
                    if asyncio.get_event_loop().time() - last_sent_time > 5:          # if more than 5 seconds 
                        if len(batch) > 0:
                            print(f"Number of elements in one batch: {len(batch)}")
                            await producer.send_batch(batch)
                            batch = await producer.create_batch()
                            last_sent_time = asyncio.get_event_loop().time()        

URL = "https://stream.wikimedia.org/v2/stream/recentchange"
header = {
        "User-Agent": "MyStreamingApp (contact: l3chster1.0@gmail.com)"
}
    
async with aiohttp.ClientSession(headers=header) as session:
    await run(session, URL) 

